In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F


spark = SparkSession.builder.appName("test").getOrCreate()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("C:/Users/Pranav/Documents/Fintech/SrcFiles/solusdt_spot_1m_jan2025.csv")

df.show()

+---+-------------+------+------+------+------+--------+
|_c0|           ts|  open|  high|   low| close|  volume|
+---+-------------+------+------+------+------+--------+
|  0|1735689600000|189.31|189.71|189.08|189.71|4904.063|
|  1|1735689660000| 189.7|189.78|189.59|189.71|1321.029|
|  2|1735689720000|189.71|190.13|189.68|190.07|7505.439|
|  3|1735689780000|190.07|190.11|189.84|189.86|1844.873|
|  4|1735689840000|189.86|189.94|189.85|189.85| 642.511|
|  5|1735689900000|189.85|190.17|189.85|190.14|1099.724|
|  6|1735689960000|190.15|190.15| 189.8|190.04|1194.558|
|  7|1735690020000|190.04|190.11|189.99|190.05| 1389.42|
|  8|1735690080000|190.05|190.08|190.01|190.08|1163.605|
|  9|1735690140000|190.08|190.08| 190.0|190.02| 507.361|
| 10|1735690200000|190.02|190.22|189.86|189.92|4856.741|
| 11|1735690260000|189.92|190.05|189.87| 190.0|2218.239|
| 12|1735690320000|190.01|190.13|189.94|190.13| 964.423|
| 13|1735690380000|190.13|190.47|190.13|190.29|3533.777|
| 14|1735690440000|190.29|190.5

In [17]:
df2 = df.withColumn(
    "ts_readable",
    from_unixtime(col("ts") / 1000)
)

df2.show(truncate=False)

+---+-------------+------+------+------+------+--------+-------------------+
|_c0|ts           |open  |high  |low   |close |volume  |ts_readable        |
+---+-------------+------+------+------+------+--------+-------------------+
|0  |1735689600000|189.31|189.71|189.08|189.71|4904.063|2025-01-01 05:30:00|
|1  |1735689660000|189.7 |189.78|189.59|189.71|1321.029|2025-01-01 05:31:00|
|2  |1735689720000|189.71|190.13|189.68|190.07|7505.439|2025-01-01 05:32:00|
|3  |1735689780000|190.07|190.11|189.84|189.86|1844.873|2025-01-01 05:33:00|
|4  |1735689840000|189.86|189.94|189.85|189.85|642.511 |2025-01-01 05:34:00|
|5  |1735689900000|189.85|190.17|189.85|190.14|1099.724|2025-01-01 05:35:00|
|6  |1735689960000|190.15|190.15|189.8 |190.04|1194.558|2025-01-01 05:36:00|
|7  |1735690020000|190.04|190.11|189.99|190.05|1389.42 |2025-01-01 05:37:00|
|8  |1735690080000|190.05|190.08|190.01|190.08|1163.605|2025-01-01 05:38:00|
|9  |1735690140000|190.08|190.08|190.0 |190.02|507.361 |2025-01-01 05:39:00|

In [28]:
w = Window.orderBy("ts").rowsBetween(-49,0)

df_ma = df2.withColumn(
            "ma_50_raw", F.avg("close").over(w)).withColumn(
                "ma_50",
                F.when(F.row_number().over(Window.orderBy("ts")) >= 50,
                    F.col("ma_50_raw"))
            )

df_ma.show(150)

+---+-------------+------+------+------+------+--------+-------------------+------------------+------------------+
|_c0|           ts|  open|  high|   low| close|  volume|        ts_readable|         ma_50_raw|             ma_50|
+---+-------------+------+------+------+------+--------+-------------------+------------------+------------------+
|  0|1735689600000|189.31|189.71|189.08|189.71|4904.063|2025-01-01 05:30:00|            189.71|              null|
|  1|1735689660000| 189.7|189.78|189.59|189.71|1321.029|2025-01-01 05:31:00|            189.71|              null|
|  2|1735689720000|189.71|190.13|189.68|190.07|7505.439|2025-01-01 05:32:00|            189.83|              null|
|  3|1735689780000|190.07|190.11|189.84|189.86|1844.873|2025-01-01 05:33:00|          189.8375|              null|
|  4|1735689840000|189.86|189.94|189.85|189.85| 642.511|2025-01-01 05:34:00|            189.84|              null|
|  5|1735689900000|189.85|190.17|189.85|190.14|1099.724|2025-01-01 05:35:00|189.

In [34]:
from pyspark.sql import functions as F, Window

w = Window.orderBy("ts")

df3 = df_ma.withColumn("prev_close", F.lag("close").over(w)).withColumn("prev_ma50", F.lag("ma_50").over(w)).withColumn(
            "buy_signal",
            F.when(
                (F.col("prev_close") < F.col("prev_ma50")) & 
                (F.col("close") > F.col("ma_50")),
                1
            ).otherwise(0)
        )
df3.filter(F.col("buy_signal") == 1).show(200)

+----+-------------+------+------+------+------+---------+-------------------+------------------+------------------+----------+------------------+----------+
| _c0|           ts|  open|  high|   low| close|   volume|        ts_readable|         ma_50_raw|             ma_50|prev_close|         prev_ma50|buy_signal|
+----+-------------+------+------+------+------+---------+-------------------+------------------+------------------+----------+------------------+----------+
| 134|1735697640000|191.01|191.26| 191.0|191.24| 3871.444|2025-01-01 07:44:00|          191.0558|          191.0558|     191.0|191.05120000000005|         1|
| 150|1735698600000|190.65| 190.9| 190.6| 190.9|  887.638|2025-01-01 08:00:00|190.86039999999997|190.86039999999997|    190.64|190.87159999999997|         1|
| 154|1735698840000|190.74|190.95| 190.7|190.95|  880.687|2025-01-01 08:04:00|190.81579999999997|190.81579999999997|    190.73|190.82099999999994|         1|
| 296|1735707360000|189.75|189.85|189.75|189.85|  46

In [36]:
df3.write.option("header", True).mode("overwrite").csv("C:/Users/Pranav/Documents/Trading_signals_jan2025.csv")